# Adversarial Minimax Framework for Autonomous Cyber Defense
This notebook contains the complete project in a single runnable file. It models network security as a dynamic, two-player zero-sum game, blending game theory proofs with deep neural networks.

We automatically download the real-world **NSL-KDD** cyber-security dataset, train an Attacker AI to perturb network packets, and simultaneously train a Defender AI to minimize the maximum classification error.


### 1. Install Dependencies

In [ ]:
!pip install pandas numpy matplotlib pyyaml torch


### 2. Data Preparation (NSL-KDD Dataset)

In [ ]:
import os
import urllib.request
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ==============================================================================
# THEORY: The NSL-KDD Dataset
# ------------------------------------------------------------------------------
# We automatically download the NSL-KDD dataset, extract the 38 continuous 
# mathematical features (like 'duration', 'src_bytes'), normalize them, and 
# load them into PyTorch DataLoaders.
# ==============================================================================

FALLBACK_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt"
TEST_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"
CATEGORICAL_COLS = [1, 2, 3]

def download_dataset(raw_dir="data"):
    os.makedirs(raw_dir, exist_ok=True)
    train_path = os.path.join(raw_dir, "KDDTrain+.txt")
    test_path = os.path.join(raw_dir, "KDDTest+.txt")
    if not os.path.exists(train_path): urllib.request.urlretrieve(FALLBACK_URL, train_path)
    if not os.path.exists(test_path): urllib.request.urlretrieve(TEST_URL, test_path)
    return train_path, test_path

def load_and_preprocess(filepath, train_mins=None, train_maxs=None):
    df = pd.read_csv(filepath, header=None)
    labels = (df[41] != 'normal').astype(int).values
    df = df.drop(columns=CATEGORICAL_COLS + [41, 42])
    features = df.values.astype(np.float32)
    
    if train_mins is None:
        train_mins = np.min(features, axis=0)
        train_maxs = np.max(features, axis=0)
    
    ranges = train_maxs - train_mins
    ranges[ranges == 0] = 1.0 
    features = (features - train_mins) / ranges
    return features, labels, train_mins, train_maxs

class CyberDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

def get_dataloaders(batch_size=512):
    train_path, test_path = download_dataset()
    train_feat, train_labels, mins, maxs = load_and_preprocess(train_path)
    test_feat, test_labels, _, _ = load_and_preprocess(test_path, train_mins=mins, train_maxs=maxs)
    
    train_loader = DataLoader(CyberDataset(train_feat, train_labels), batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(CyberDataset(test_feat, test_labels), batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, train_feat.shape[1]


### 3. Neural Network Architectures

In [ ]:
import torch.nn as nn

# ==============================================================================
# THEORY: The Networks
# ------------------------------------------------------------------------------
# Defender (D_theta): Minimizes classification error. Outputs probability of Attack (0 to 1).
# Attacker (A_phi): Maximizes classification error. Outputs bounded perturbations [-1, 1] using Tanh.
# ==============================================================================

class DefenderNet(nn.Module):
    def __init__(self, input_dim):
        super(DefenderNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.network(x)

class AttackerPolicy(nn.Module):
    def __init__(self, input_dim):
        super(AttackerPolicy, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, input_dim), nn.Tanh() 
        )
    def forward(self, x): return self.network(x)

class CyberGameEnv:
    def __init__(self, epsilon=0.05):
        self.epsilon = epsilon
    def apply_attack(self, original_logs, perturbations):
        return torch.clamp(original_logs + (perturbations * self.epsilon), 0.0, 1.0)


### 4. Minimax Training Loop

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

def train_minimax_step(batch_x, batch_y, attacker, defender, env, opt_a, opt_d, criterion):
    malicious_mask = (batch_y == 1.0).flatten()
    real_malicious = batch_x[malicious_mask]
    
    if real_malicious.size(0) == 0:
        opt_d.zero_grad()
        loss = criterion(defender(batch_x), batch_y)
        loss.backward()
        opt_d.step()
        return 0.0, loss.item()

    target_malicious = torch.ones(real_malicious.size(0), 1, device=batch_x.device)
    target_evaded = torch.zeros(real_malicious.size(0), 1, device=batch_x.device) 

    # PHASE 1: Optimize Attacker
    for p in defender.parameters(): p.requires_grad = False
    for p in attacker.parameters(): p.requires_grad = True

    opt_a.zero_grad()
    perturbations = attacker(real_malicious)
    attacked_logs = env.apply_attack(real_malicious, perturbations)
    attacker_loss = criterion(defender(attacked_logs), target_evaded)
    attacker_loss.backward()
    opt_a.step()

    # PHASE 2: Optimize Defender
    for p in defender.parameters(): p.requires_grad = True
    for p in attacker.parameters(): p.requires_grad = False

    opt_d.zero_grad()
    loss_standard = criterion(defender(batch_x), batch_y)
    with torch.no_grad():
        worst_case = env.apply_attack(real_malicious, attacker(real_malicious))
    loss_robust = criterion(defender(worst_case), target_malicious)
    
    defender_loss = (loss_standard + loss_robust) / 2
    defender_loss.backward()
    opt_d.step()

    return attacker_loss.item(), defender_loss.item()


### 5. Execution & Evaluation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, test_loader, feat_dim = get_dataloaders()

defender = DefenderNet(feat_dim).to(device)
attacker = AttackerPolicy(feat_dim).to(device)
env = CyberGameEnv(epsilon=0.05) 

opt_d = optim.AdamW(defender.parameters(), lr=1e-3, weight_decay=1e-4)
opt_a = optim.AdamW(attacker.parameters(), lr=2e-3, weight_decay=1e-4) 
criterion = nn.BCELoss()

EPOCHS = 15
for epoch in range(1, EPOCHS + 1):
    defender.train()
    attacker.train()
    for batch_x, batch_y in train_loader:
        train_minimax_step(batch_x.to(device), batch_y.to(device), attacker, defender, env, opt_a, opt_d, criterion)
    print(f"Epoch {epoch} complete.")

# Evaluation
defender.eval()
attacker.eval()
correct_static, correct_adaptive, total_mal = 0, 0, 0

for batch_x, batch_y in test_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    mask = (batch_y == 1.0).flatten()
    malicious = batch_x[mask]
    if malicious.size(0) == 0: continue
        
    total_mal += malicious.size(0)
    correct_static += (defender(malicious) >= 0.5).int().sum().item()
    correct_adaptive += (defender(env.apply_attack(malicious, attacker(malicious))) >= 0.5).int().sum().item()

acc_static = correct_static / total_mal * 100
acc_adaptive = correct_adaptive / total_mal * 100

print(f"\nStatic Accuracy: {acc_static:.2f}% | Robust Accuracy: {acc_adaptive:.2f}%")

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax1 = plt.subplots(figsize=(8, 6))

# Plot 1: Robustness Bar Chart
bars = ax1.bar(["Static (Clean Data)", "Adaptive (Active Attack)"], [acc_static, acc_adaptive], color=['#1f77b4', '#d62728'], width=0.6)
ax1.set_ylim(0, 100)
ax1.set_ylabel("Defender True Positive Rate (%)")
ax1.set_title("Adversarial Robustness")
for bar in bars:
    h = bar.get_height()
    ax1.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h/2), xytext=(0,0), textcoords="offset points", ha='center', va='center', color='white', fontweight='bold', size=14)

plt.show()


### 6. Feature Perturbation Strategy

In [ ]:
import numpy as np

# Get one batch of malicious traffic
sample_traffic = None
for batch_x, batch_y in test_loader:
    mask = (batch_y == 1.0).flatten()
    if mask.sum() > 0:
        sample_traffic = batch_x[mask].to(device)
        break

with torch.no_grad():
    perturbations = attacker(sample_traffic).cpu().numpy()

key_indices = [0, 4, 5, 22, 23, 31, 32, 33]
key_labels = ['Duration', 'Src Bytes', 'Dst Bytes', 'Count', 'Srv Count', 'Dst Host Count', 'Dst Srv Count', 'Same Srv Rate']
avg_pert = np.abs(perturbations[:, key_indices].mean(axis=0))

N = len(key_labels)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
avg_pert = np.concatenate((avg_pert, [avg_pert[0]]))

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], key_labels, size=12, fontweight='bold')
ax.plot(angles, avg_pert, linewidth=2, linestyle='solid', color='#d62728')
ax.fill(angles, avg_pert, '#d62728', alpha=0.25)
plt.title("Attacker Policy: Feature Perturbation Strategy", size=18, y=1.1)
plt.show()
